# Notebook 03: DPO Trials (5 runs)

**Purpose:** Take the winning SFT model and run 5 DPO preference fine-tuning trials. Pick the winner.

**Inputs:**
- `results/sft_winner.json` (pointer to winning SFT trial)
- `models/sft_<winner>/` (the winning SFT LoRA adapter)
- `data/dpo_pairs.json` (~150 preference pairs from notebook 01)
- `configs/dpo_trials.json` (5 DPO trial configs)

**Outputs:** `results/dpo_trial_*.json` + LoRA adapters in `models/dpo_trial_*/`

**Runtime:** ~2-5 hours on Kaggle T4 for all 5 DPO trials.


## Step 1: Setup

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import torch

KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path("/kaggle/working/daa-helper") if KAGGLE else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))

if KAGGLE:
    import subprocess
    for pkg in ["transformers>=4.45.0", "peft>=0.13.0", "trl>=0.11.0",
                "bitsandbytes>=0.44.0", "accelerate>=1.0.0", "datasets>=3.0.0",
                "sacrebleu", "bert-score"]:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

from utils.io_helpers import load_json, save_trial_result, list_existing_results
from utils.evaluation import run_inference_on_prompts, evaluate_responses, free_memory

from huggingface_hub import login
if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)

HF_USERNAME = "YOUR_HF_USERNAME"  # <-- EDIT THIS
print("Setup complete.")


## Step 2: Load winner pointer, configs, prompts, gold answers, DPO data

In [ ]:
winner_info = load_json("sft_winner.json", base_dir="results")
SFT_WINNER_NAME = winner_info["winning_trial"]
SFT_WINNER_DIR = str(PROJECT_ROOT / "models" / f"sft_{SFT_WINNER_NAME}")
print(f"SFT winner: {SFT_WINNER_NAME}")
print(f"Adapter dir: {SFT_WINNER_DIR}")
assert Path(SFT_WINNER_DIR).exists(), f"Winner adapter not found at {SFT_WINNER_DIR}"

dpo_config = load_json("dpo_trials.json", base_dir="configs")
prompts_data = load_json("test_prompts.json", base_dir="data")
gold_data = load_json("gold_answers.json", base_dir="data")
dpo_pairs_data = load_json("dpo_pairs.json", base_dir="data")

prompts = [p["prompt"] for p in prompts_data["prompts"]]
gold_answers = [a["gold_answer"] for a in gold_data["answers"]]
dpo_trials = dpo_config["trials"]
pairs = dpo_pairs_data["pairs"]

print(f"\nLoaded {len(dpo_trials)} DPO trial configs")
print(f"Loaded {len(pairs)} preference pairs for DPO training")


## Step 3: Prepare DPO dataset (HuggingFace Dataset format)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer for chat templating
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM = ("You are a Design and Analysis of Algorithms (DAA) tutor. "
          "Guide the student through the problem step by step. "
          "Help them understand the reasoning rather than just giving the final answer "
          "unless explicitly asked for the solution.")


def format_pair(pair):
    """Format a preference pair for TRL DPO trainer."""
    prompt_messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": pair["prompt"]}
    ]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    return {
        "prompt": prompt_text,
        "chosen": pair["chosen"],
        "rejected": pair["rejected"]
    }


formatted = [format_pair(p) for p in pairs]
# Split 90/10 train/eval
split_idx = int(0.9 * len(formatted))
dpo_train_ds = Dataset.from_list(formatted[:split_idx])
dpo_eval_ds = Dataset.from_list(formatted[split_idx:])

print(f"DPO train: {len(dpo_train_ds)} pairs")
print(f"DPO eval:  {len(dpo_eval_ds)} pairs")
print(f"\nSample formatted pair (prompt):")
print(dpo_train_ds[0]["prompt"][:500])


## Step 4: Define DPO training function

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model
from trl import DPOTrainer, DPOConfig


def load_sft_winner_for_dpo():
    """Load the SFT winner adapter merged onto the quantized base, ready for DPO."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    model = PeftModel.from_pretrained(base, SFT_WINNER_DIR, is_trainable=True)
    return model


def run_dpo_trial(trial_config, train_dataset, eval_dataset, output_dir):
    """Run one DPO trial."""
    print(f"\n{'='*70}\nDPO TRIAL: {trial_config['name']}\n{'='*70}")
    print(f"Description: {trial_config['description']}")

    model = load_sft_winner_for_dpo()

    # For DPO, we add a NEW LoRA on top of the existing SFT adapter
    # (alternative: merge SFT adapter into base, then add new LoRA — but that loses 4-bit savings)
    # TRL's DPOTrainer accepts a peft_config to add new LoRA params, OR we can train the existing adapter further.
    # Simpler path: train the existing SFT LoRA further with DPO loss.

    dpo_args = DPOConfig(
        output_dir=output_dir,
        num_train_epochs=trial_config["num_train_epochs"],
        per_device_train_batch_size=trial_config["per_device_train_batch_size"],
        gradient_accumulation_steps=trial_config["gradient_accumulation_steps"],
        per_device_eval_batch_size=trial_config["per_device_train_batch_size"],
        learning_rate=trial_config["learning_rate"],
        beta=trial_config["beta"],
        max_length=trial_config["max_length"],
        max_prompt_length=trial_config["max_prompt_length"],
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=20,
        save_strategy="no",
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        bf16=True,
        optim="paged_adamw_8bit",
        report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,
    )

    trainer = DPOTrainer(
        model=model,
        args=dpo_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
    )

    start = time.time()
    train_result = trainer.train()
    elapsed = time.time() - start

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    eval_metrics = trainer.evaluate()

    train_metrics = {
        "train_loss": train_result.training_loss,
        "eval_loss": eval_metrics.get("eval_loss"),
        "rewards_chosen": eval_metrics.get("eval_rewards/chosen"),
        "rewards_rejected": eval_metrics.get("eval_rewards/rejected"),
        "rewards_margin": eval_metrics.get("eval_rewards/margins"),
        "train_runtime_seconds": elapsed,
        "global_step": train_result.global_step,
    }

    del trainer, model
    free_memory()

    return output_dir, train_metrics


def evaluate_dpo_adapter(adapter_dir, prompts, gold_answers):
    """Load DPO adapter on top of base, run inference, score."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()

    responses = run_inference_on_prompts(model, tokenizer, prompts, max_new_tokens=512, temperature=0.7)
    eval_results = evaluate_responses(responses, gold_answers)

    sample_responses = [{"prompt": p, "response": r, "gold": g}
                       for p, r, g in zip(prompts, responses, gold_answers)]

    del model, base
    free_memory()
    return eval_results, sample_responses


## Step 5: Run all 5 DPO trials

In [ ]:
existing = set(list_existing_results(stage="dpo"))
print(f"Existing DPO results: {existing}")

for trial in dpo_trials:
    result_filename = f"dpo_{trial['name']}.json"
    if result_filename in existing:
        print(f"\n✓ Skipping {trial['name']} (already completed)")
        continue

    output_dir = str(PROJECT_ROOT / "models" / f"dpo_{trial['name']}")
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    try:
        adapter_dir, train_metrics = run_dpo_trial(trial, dpo_train_ds, dpo_eval_ds, output_dir)

        print(f"\nEvaluating on 10 test prompts...")
        eval_results, sample_responses = evaluate_dpo_adapter(adapter_dir, prompts, gold_answers)

        save_trial_result(
            trial_name=trial["name"],
            stage="dpo",
            config=trial,
            eval_results=eval_results,
            train_metrics=train_metrics,
            sample_responses=sample_responses
        )

        print(f"\n✓ {trial['name']} complete.")
        print(f"  Mean BLEU:         {eval_results['aggregate']['mean_bleu']:.2f}")
        print(f"  Mean BERTScore F1: {eval_results['aggregate']['mean_bertscore_f1']:.4f}")
        print(f"  Combined score:    {eval_results['aggregate']['combined_score']:.4f}")
        print(f"  Eval loss:         {train_metrics['eval_loss']:.4f}")
        print(f"  Reward margin:     {train_metrics.get('rewards_margin', 'n/a')}")
        print(f"  Time:              {train_metrics['train_runtime_seconds']:.0f}s")

    except Exception as e:
        print(f"\n✗ ERROR in {trial['name']}: {e}")
        import traceback
        traceback.print_exc()
        free_memory()
        continue

print("\n\nAll DPO trials processed.")


## Step 6: Pick the winning DPO trial

In [ ]:
results = []
for trial in dpo_trials:
    try:
        r = load_json(f"dpo_{trial['name']}.json", base_dir="results")
        results.append(r)
    except FileNotFoundError:
        print(f"⚠ Missing result for {trial['name']}")

def sort_key(r):
    score = r["evaluation"]["aggregate"]["combined_score"]
    eval_loss = r["training_metrics"].get("eval_loss") or float("inf")
    return (-score, eval_loss)

results.sort(key=sort_key)

print("="*70)
print("DPO TRIAL RANKING (best first)")
print("="*70)
print(f"{'Trial':<30} {'BLEU':>8} {'BERTScore':>10} {'Combined':>10} {'EvalLoss':>10}")
for r in results:
    name = r["trial_name"]
    agg = r["evaluation"]["aggregate"]
    eval_loss = r["training_metrics"].get("eval_loss", float("nan"))
    print(f"{name:<30} {agg['mean_bleu']:>8.2f} {agg['mean_bertscore_f1']:>10.4f} {agg['combined_score']:>10.4f} {eval_loss:>10.4f}")

winner = results[0]
print(f"\n🏆 DPO WINNER: {winner['trial_name']}")
print(f"   Config: {winner['config']}")

with open(PROJECT_ROOT / "results" / "dpo_winner.json", "w") as f:
    json.dump({"winning_trial": winner["trial_name"],
               "winning_config": winner["config"],
               "winning_metrics": winner["evaluation"]["aggregate"]}, f, indent=2)
print(f"\nSaved pointer to results/dpo_winner.json")
print("\n✓ Notebook 03 complete. Run notebook 04 for final comparison.")
